# task 1: pre processing before mining & graph

import

In [17]:
import pandas as pd
import json
import os

save paths

In [18]:
RAW_DATA_PATH = "../data/raw/"
PROCESSED_DATA_PATH = "../data/processed_transactions/"

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

load data

In [19]:
meta_chunks = pd.read_json(os.path.join(RAW_DATA_PATH, 'meta_Electronics.jsonl'), 
                           lines=True, chunksize=50000)

meta_list = []
for chunk in meta_chunks:
    if 'title' in chunk.columns and 'parent_asin' in chunk.columns:
        meta_list.append(chunk[['parent_asin', 'title']])

df_meta = pd.concat(meta_list).drop_duplicates('parent_asin')

print(f" Full Metadata loaded. Total unique product titles: {len(df_meta)}")

 Full Metadata loaded. Total unique product titles: 1610012


In [20]:
df_electronics = pd.read_json(os.path.join(RAW_DATA_PATH, 'Electronics.jsonl'), 
                             lines=True, nrows=30000)

df_electronics = df_electronics[['user_id', 'parent_asin']]

print(f"Loaded {len(df_electronics)} interactions.")

Loaded 30000 interactions.


merging , cleaning metadat

In [21]:
merged_df = pd.merge(df_electronics[['user_id', 'parent_asin']], df_meta, on='parent_asin', how='inner')

merged_df = merged_df.dropna(subset=['title'])

merged_df = merged_df[merged_df['title'].str.strip() != ""]

print(f" Data merged: {len(merged_df)}")


 Data merged: 30000


# grouping :  Transform to Transaction Format 

Creating the Transaction Basket

In [22]:
basket = merged_df.groupby('user_id')['title'].apply(list).reset_index()

Removing duplicate items within the same transaction

In [23]:
basket['title'] = basket['title'].apply(lambda x: list(set(x)))


 Filter Transactions: Keep only those with more than 1 item

In [24]:
basket = basket[basket['title'].map(len) > 1]

print(f"Total transactions for mining: {len(basket)}")

Total transactions for mining: 2966


final check

In [25]:
# 1. Check if the basket is empty
if len(basket) == 0:
    print("  The basket is empty!")
else:
    avg_items = basket['title'].map(len).mean()
    max_items = basket['title'].map(len).max()
    
    print(f" Total Transactions for Mining: {len(basket)}")
    print(f" Average items per basket: {avg_items:.2f}")
    print(f" Max items in a single basket: {max_items}")
    

    print("\nSample of Cleaned Transactions:")
    print(basket.head(5))


 Total Transactions for Mining: 2966
 Average items per basket: 9.63
 Max items in a single basket: 567

Sample of Cleaned Transactions:
                        user_id  \
0  AE22CFXT3QZKUQJORVTGL3VQXAAA   
1  AE23NAHINBRGBQ3A46YME3TPRL3A   
2  AE257SG5NR3BZMRYIVVY7YW7UVBQ   
3  AE25NQAZI3725GZIL5FS52ZIKWKQ   
4  AE26ICWKMFJEHDH5VDX4W42H2NMA   

                                               title  
0  [HminSen Case for Lenovo Tab M10 FHD Plus TB-X...  
1  [Dteck Case for All-New Fire HD 10 & HD 10 Plu...  
2  [Headphone Stand Hanger, JOTO Silicone Under D...  
3  [Laptop Stand, Alloyseed Foldable Laptop Table...  
4  [Phone Tripod Tall, Extend to 82 Inch Tripod w...  


save data


Pickle saves the transactions as actual Python Lists but CSV , Python will read each transaction as single long String (text), making it harder to process for Association Rule Mining.

In [26]:
basket.to_pickle(os.path.join(PROCESSED_DATA_PATH, 'cleaned_transactions.pkl'))
basket.to_csv(os.path.join(PROCESSED_DATA_PATH, 'inspected_transactions.csv'), index=False)

print(f"Files saved to {PROCESSED_DATA_PATH}")

Files saved to ../data/processed_transactions/
